In [2]:
import sys

project_root = "/home/latifi/proyectos/housing_api"
# Añadir al sys.path si no está ya
if project_root not in sys.path:
    sys.path.append(project_root)

from app.models.listing_raw import listing_raw

In [ ]:
from bs4 import Tag , ResultSet
import re


class dto_builder:
    @staticmethod
    def get_text(item: Tag, selector: str):
        element = item.select_one(selector)
        return element.get_text(strip=True) if element else None
    
    @staticmethod
    def get_number(item: Tag, selector: str):
        text = dto_builder.get_text(item,selector)
        if(text is None): return None
        cleaned = re.sub(r"[^0-9.,]", "", text)
        cleaned = cleaned.replace(",", ".")
        parts = cleaned.split(".")
        if len(parts) > 2:
            cleaned = parts[0] + "." + "".join(parts[1:])
        try:
            number = float(cleaned)
            return int(number) if number.is_integer() else number
        except ValueError:
            return None

    @staticmethod
    def create_list_from_pisos_com(items:ResultSet[Tag],region:str):
        dtos:list[listing_raw] = []
        for item in items:
            dto = dto_builder.create_dto_from_pisos_com(item)
            dto.region = region
            dtos.append(dto)
        
        return dtos

    @staticmethod
    def create_dto_from_pisos_com(item: Tag) -> listing_raw:
        PATH_CLASSNAME = ".p-sm.ad-preview__subtitle"
        PRICE_CLASSNAME = ".ad-preview__price"
        SQUARE_METERS = ".ad-preview__char:nth-child(3)"
        ROOMS = ".ad-preview__char:nth-child(1)"
        BATHROOMS = ".ad-preview__char:nth-child(1)"
        DESCRIPTION = ".ad-preview__description"

        direccion = dto_builder.get_text(item, PATH_CLASSNAME)
        precio = dto_builder.get_number(item, PRICE_CLASSNAME)
        m2 = dto_builder.get_number(item, SQUARE_METERS)
        habitaciones = dto_builder.get_number(item, ROOMS)
        banos = dto_builder.get_number(item, BATHROOMS)
        descripcion = dto_builder.get_text(item, DESCRIPTION)

        dto = listing_raw(
            portal="pisos.com",
            precio=precio,
            m2=m2,
            habitaciones=habitaciones,
            banos=banos,
            descripcion=descripcion,
            direccion=direccion
        )

        return dto


        

# Escraping en FotoCasa

In [ ]:
import requests
from bs4 import BeautifulSoup



import unicodedata
import re
import math

def limpiar_texto(texto: str) -> str:
    texto = texto.lower()
    texto_normalizado = unicodedata.normalize("NFD", texto)
    texto_normalizado = (
        texto_normalizado
        .replace("ñ", "n")
    )
    texto_sin_acentos = "".join(
        c for c in texto_normalizado
        if unicodedata.category(c) != "Mn"
    )
    texto_sin_espacios = re.sub(r"\s+", "_", texto_sin_acentos)
    return texto_sin_espacios

def scraping_pisos_all_pag(region:str="Miño"):
    listing_raws:list[listing_raw] = []
    URL = f"https://www.pisos.com/venta/pisos-{limpiar_texto(region)}/"
    HEADERS = {"User-Agent": "Mozilla/5.0"}
    html = requests.get(URL, headers=HEADERS).text
    soup = BeautifulSoup(html, "lxml")
    text = soup.select_one("span.pagination__counter").get_text().replace(".","")
    pages = math.ceil(float(get_second_number(text)) / 30)
    for page in range(pages):
        listing_raws += scrapig_pisos_pag(region,page+1)
    return listing_raws

def scrapig_pisos_pag(region:str,page:int):
    URL = f"https://www.pisos.com/venta/pisos-{limpiar_texto(region)}/{page}"
    HEADERS = {"User-Agent": "Mozilla/5.0"}
    CAROUSEL_CLASSNAME = ".ad-preview"
    html = requests.get(URL, headers=HEADERS).text
    soup = BeautifulSoup(html, "lxml")
    items = soup.select(CAROUSEL_CLASSNAME)
    return dto_builder.create_list_from_pisos_com(items,region)
    
    
def get_second_number(text:str):
    number = re.findall(r"\d+", text)
    second = int(number[1]) if len(number) > 1 else None
    return second


houses = scraping_pisos_all_pag()


In [4]:
houses[1].to_dict()

{'id': None,
 'portal': 'pisos.com',
 'region': 'Miño',
 'direccion': 'A Carreira (Miño)',
 'descripcion': 'Casa con amplia finca en miño\n\nBienvenido a esta extraordinaria casa en miño, a tan solo 5 minutos de la playa y a 10 minutos en c...',
 'm2': 339,
 'precio': 570.0,
 'habitaciones': 7,
 'banos': 7,
 'tipo_vivienda': None,
 'ascensor': None,
 'garaje': None,
 'source_link': None}

# Orquestrador de scraping

In [ ]:
import unicodedata
import requests
from bs4 import BeautifulSoup, Tag
from collections.abc import Callable
import re


class path_setting:
    def __init__(
        self,
        propety_name: str,
        path: str,
        get_method: Callable[[Tag, str], str | int] | None = None,
    ):
        self.path = path
        self.propety_name = propety_name
        if get_method is not None:
            self.get_method = get_method
        else:
            self.get_method = scraping_client.get_text

    def set_propety(self, obj: object, card: Tag):
        value = self.get_method(card, self.path)
        setattr(obj, self.propety_name, value)


class path_settings:

    def __init__(
        self,
        card_path: str,
        items: list[path_setting] = [],
    ):
        self.__items: list[path_setting] = items
        self.card_path = card_path

    def add(self, item: path_setting):
        self.__items.append(item)

    def scraping(self, obj: object, card: Tag):
        for path in self.__items:
            path.set_propety(obj, card)


class path_settings_old:
    def __init__(
        self,
        card_path: str,
        path_classname: str,
        price_classname: str,
        bathrooms: str,
        square_meters: str,
        rooms: str,
        description: str,
        source_link: str,
    ):
        self.card_path = card_path
        self.path_classname = path_classname
        self.price_classname = price_classname
        self.square_meters = square_meters
        self.bathrooms = bathrooms
        self.rooms = rooms
        self.description = (description,)
        self.source_link = source_link


class scraping_settings:
    def __init__(
        self, url: str,
        paths: path_settings, 
        get_source_link: Callable[[Tag], str],
        get_info: Callable[[str], tuple[str, int]]
    ):

        self.validate_url(url)
        self.get_source_link = get_source_link
        self.url = url
        self.paths = paths
        self.get_info = get_info

    def validate_url(self, url: str):
        if not re.match(r"^https?://[^\s]+$", url):
            raise ValueError("No has proporcionado una URL válida")

        if "{query}" not in url:
            raise ValueError("La URL debe incluir '{query}'")

        if "{page}" not in url:
            raise ValueError("La URL debe incluir '{page}'")


class scraping_client:
    def __init__(self, portal: str, conf: scraping_settings):
        self.__portal = portal
        self.__conf = conf
        self.header = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/117.0",
            "Accept-Language": "es-ES,es;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Connection": "keep-alive",
        }

    def __build_url(self, query: str, page=1):
        url: str = self.__conf.url.format(query=query, page=page)
        return url

    def get_adds(self, query: str):
        adds: list[listing_raw] = list()
        info = self.__conf.get_info(query)

        if info is None:
            print(f"No se encontro ninguna pagina de {query}")
        query = info[0]
        pages = info[1]
        for page in range(pages):
            adds += self.__get_adds_from_page(query, page + 1)
        return adds

    def __get_adds_from_page(self, query: str, page=1):
        url = self.__build_url(query, page)

        html = requests.get(url, headers=self.header).text
        return self.__get_adds_from_htm(html)

    def __get_adds_from_htm(self, html: str):
        soup = BeautifulSoup(html, "html.parser")
        cards = soup.select(self.__conf.paths.card_path)
        with open("hola.html", "w") as f:
            f.write(html)
        adds: list[listing_raw] = list()
        for card in cards:
            url = self.__conf.get_source_link(card)
            if(url is None):
                continue
            result = requests.get(url).text
            soup = BeautifulSoup(result).select_one("body")
            dto = self.__create_dto_by_card(soup)
            dto.source_link = url
            adds.append(dto)
        return adds

    @staticmethod
    def clean_text(text: str):
        text = text.lower()
        text = str.replace(text, " ", "-")
        text = str.replace(text, "ñ", "n")
        normalitated = unicodedata.normalize("NFD", text)
        return normalitated

    @staticmethod
    def get_text_from_attribute(item: Tag, selector: str, attribute: str):
        element = item.select_one(selector)
        if element is None:
            return None
        return element.get(attribute)

    @staticmethod
    def get_text(item: Tag, selector: str):
        element = item.select_one(selector)
        return element.get_text(strip=True) if element else None

    @staticmethod
    def get_number(item: Tag, selector: str):
        text = scraping_client.get_text(item, selector)
        if text is None:
            return None
        cleaned = re.sub(r"[^0-9.,]", "", text)
        cleaned = cleaned.replace(",", ".")
        parts = cleaned.split(".")
        if len(parts) > 2:
            cleaned = parts[0] + "." + "".join(parts[1:])
        try:
            number = float(cleaned)
            return int(number) if number.is_integer() else number
        except ValueError:
            return None

    def __create_dto_by_card(self, card: Tag) -> listing_raw:
        dto = listing_raw(portal=self.__portal)
        self.__conf.paths.scraping(dto, card)

        return dto

In [4]:
import requests
def get_item(items):
    for item in items:
        if item.get("type") == 2:
            return item
    return None
        
def get_info_indomio(query: str)->tuple[str,int]:
    url1 = f"https://www.indomio.es/api-next/geography/autocomplete/?macrozones=1&query={query}&__lang=es"
    result = requests.get(url1).json()
    result = get_item(result)
    
    if result is None:
        return None;
    keyURL = result["keyurl"]
    id = result["id"]
    url2 = f" https://www.indomio.es/api-next/search-list/listings/?fkRegione=es-12&idProvincia=es-12-15&idComune={id}&idNazione=S&idContratto=1&idCategoria=1&__lang=es&pag=1&paramsCount=0&path=%2Fventa-casas%2Fa-coruna-capital%2F"
    result = requests.get(url2).json()
    return keyURL , result["totalAds"]



get_info_indomio("miño")


('mino', 27)

# Scraping en FotoCasa

In [5]:
import math
def get_info_fotocasa(query: str)->tuple[str,int] | None:
    url1 = f" https://search.gw.fotocasa.es/v2/suggest"
    payload = {
        "query": query,
        "filters": {"propertyTypeFilter": "HOME", "transactionTypeFilter": "SALE"},
    }


    result = requests.post(url1,json=payload).json()
    
    if len(result) <= 0:
        return None
    result = result[0]
    
    keyURL = scraping_client.clean_text(result["baseText"])
    adds = result["adsCount"]
    return keyURL , math.ceil(adds/30)


get_info_fotocasa("miño")

('mino', 2)

In [7]:
# fotocasa_paths = path_settings_old(
#     card_path="article",
#     bathrooms="ul.text-body-1 li:nth-child(2)",
#     description="[data-panot-component='collapsible-content'] p",
#     path_classname="h3 a",
#     price_classname=".text-display-3 span:first-child",
#     rooms="ul.text-body-1 li:nth-child(1)",
#     square_meters="ul.text-body-1 li:nth-child(3)"
# )


def get_url_pisos_com(card:Tag):
    PATH = "h3 a"
    link = scraping_client.get_text_from_attribute(card,PATH,"href")
    if(link is None):
        return None
    return "https://www.fotocasa.es" + str(link)


fotocasa_path_settings = path_settings(
    card_path="article",
    items=[
    # path_setting("habitaciones", ".re-DetailFeatureGrid .sui-LayoutGrid-item:nth-child(2) .re-SharedFeature-legendDescription", scraping_client.get_number),
    # path_setting("m2", ".re-DetailFeatureGrid .sui-LayoutGrid-item:nth-child(4) .re-SharedFeature-legendDescription", scraping_client.get_number),
    path_setting("precio", "span.re-DetailHeader-price", scraping_client.get_number),
])
fotocasa_settings = scraping_settings("https://www.fotocasa.es/es/comprar/viviendas/{query}/todas-las-zonas/l/{page}",fotocasa_path_settings,get_url_pisos_com,get_info_fotocasa)
fotocasa_clients = scraping_client("fotocasa",fotocasa_settings)
casas = fotocasa_clients.get_adds("miño")

casas[0].to_dict()

KeyboardInterrupt: 

# Scraping em Pisos.com

In [118]:
import math
import json

def get_info_pisos_com(query: str)->tuple[str,int] | None:
    url1 = f"https://www.pisos.com/FilterGeo/GetSuggestedGeos?word={query}&searchContext=0.1000.F002.M00000000015048.0.0X0.0.0.0X0.0.0X0X0.1.0.0X0.0X0.0.0"


    result = requests.get(url1)
    
    # if len(result) <= 0:
        # return None
    result = json.loads(result.text.strip())
    result = json.loads(result)[0]
    
    keyURL = scraping_client.clean_text(result["UrlName"])
    adds = result["Contador"]
    return keyURL , math.ceil(adds/30)


get_info_pisos_com("ferrol")

('area_de_ferrol', 38)

In [ ]:
fotocasa_paths = path_settings_old(
    card_path="div.ad-preview__bottom",
    bathrooms=".ad-preview__char:nth-of-type(2)",
    description=".ad-preview__description",
    path_classname=".ad-preview__title",
    price_classname=".ad-preview__price",
    rooms=".ad-preview__char:nth-of-type(1)",
    square_meters=".ad-preview__char:nth-of-type(3)"
)

fotocasa_settings = scraping_settings("https://www.pisos.com/venta/pisos-{query}/{page}",fotocasa_paths,get_info_pisos_com)
fotocasa_clients = scraping_client("pisos.com",fotocasa_settings)
casas = fotocasa_clients.get_adds("ferrol")
casas[0].to_dict()

{'id': None,
 'portal': 'pisos.com',
 'region': None,
 'direccion': 'Piso en Carretera de Castela, 384',
 'descripcion': 'Desde inmobiliaria freixeiro os presentamos esta preciosa vivienda con ascensor en plena carretera de castilla, con todos los serv...',
 'm2': 112,
 'precio': 115.0,
 'habitaciones': 3,
 'banos': 2,
 'tipo_vivienda': None,
 'ascensor': None,
 'garaje': None}